# 9. Supervisor Multi-Agent System
**Industry:** Manufacturing

Build a hierarchical multi-agent system where a 'Supervisor' agent routes requests to specialized agents (Researcher, Coder, Enhancer) and validates the final output.

In [ ]:
!pip install langgraph langchain langchain-google-genai pydantic langchain_experimental

In [ ]:
from typing import TypedDict, Literal
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel
from langgraph.graph import StateGraph, START, END

class Router(BaseModel):
    next_node: Literal["Researcher", "Coder", "FINISH"]

llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash")
router_llm = llm.with_structured_output(Router)

class State(TypedDict):
    query: str
    research_data: str
    code_data: str
    final_answer: str

def supervisor(state: State):
    prompt = f"You are a Supervisor. Next steps?\nQuery: {state['query']}\nResearch: {state.get('research_data', 'None')}\nCode: {state.get('code_data', 'None')}"
    decision = router_llm.invoke(prompt)
    return {"next_node": decision.next_node}

def researcher(state: State):
    print("-> Researcher working...")
    return {"research_data": "Found info: Sensor 3 is prone to overheating when voltage > 5V."}

def coder(state: State):
    print("-> Coder working...")
    return {"code_data": "def check_voltage(v): return v > 5"}

workflow = StateGraph(State)
workflow.add_node("Supervisor", supervisor)
workflow.add_node("Researcher", researcher)
workflow.add_node("Coder", coder)

workflow.add_edge(START, "Supervisor")
workflow.add_conditional_edges("Supervisor", lambda state: state.get("next_node"), {
    "Researcher": "Researcher",
    "Coder": "Coder",
    "FINISH": END
})
workflow.add_edge("Researcher", "Supervisor")
workflow.add_edge("Coder", "Supervisor")

app = workflow.compile()

query = "research causes of downtime on Line 3 and write code to analyze the sensor logs"
for event in app.stream({"query": query}):
    pass # Log outputs are inside nodes